# 05 Strategy Backtest

Turn selected factors into a composite signal, run portfolio backtests, and compare daily NAV paths across methods.

In [ ]:
from pathlib import Path
import sys

import pandas as pd


def locate_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "apps").exists():
            return candidate
    raise RuntimeError("repo root not found")


REPO_ROOT = locate_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from apps.quant_platform.research.data_loader import ResearchDataLoader
from apps.quant_platform.research.factor_engine.composite import CompositeFactorBuilder
from apps.quant_platform.research.strategy import FactorStrategy

RESEARCH_ROOT = REPO_ROOT / "apps/quant_platform/research"
OUTPUT_ROOT = RESEARCH_ROOT / "output/notebook_backtest"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
loader = ResearchDataLoader()
builder = CompositeFactorBuilder()

In [ ]:
panel = loader.prepare_panel(
    loader.load_panel(
        start_date="2024-01-01",
        end_date="2024-06-30",
        columns=[
            "ts_code", "trade_date", "open", "close", "pct_chg", "turnover_rate_f",
            "volume_ratio", "pe_ttm", "pb", "ps_ttm",
        ],
    )
)
factor_cols = ["pct_chg", "turnover_rate_f", "volume_ratio", "pe_ttm", "pb"]
panel.head()

In [ ]:
strategy = FactorStrategy()
results = {}
for method in ["equal_weight", "ic_weighted", "ml"]:
    composite = builder.build(panel, factor_cols=factor_cols, target_col="overnight_return", method=method)
    results[method] = strategy.run(composite, factor_col="composite_factor", return_col="overnight_return")

summary = pd.DataFrame(
    [
        {
            "method": method,
            **payload["summary"],
        }
        for method, payload in results.items()
    ]
).sort_values("sharpe_ratio", ascending=False)
summary.to_csv(OUTPUT_ROOT / "05_strategy_summary.csv", index=False)
summary

In [ ]:
nav_frame = pd.concat(
    [payload["daily_results"]["nav"].rename(method) for method, payload in results.items()],
    axis=1,
)
nav_frame.to_csv(OUTPUT_ROOT / "05_strategy_nav.csv")
nav_frame.plot(figsize=(10, 5), title="Composite strategy NAV comparison")

## Review Points

- Prefer the method that balances Sharpe, drawdown, and turnover rather than chasing final NAV alone.
- If ML wins only by taking much higher turnover, revisit trading-cost assumptions and constraint realism.
- Exported NAV and summary CSVs can be compared directly against the full pipeline reports.